# Track 4 Bootstrap Model Comparison — Colab Execution Harness

Implements `host_software/ml_jetson_vla/docs/BOOTSTRAP_MODEL_COMPARISON_PLAN.md` (item 4's
plan; **read that doc in full before touching this notebook** — this is the execution
harness for it, not a restatement of it). This is Track 4 / Phase 6 item 5.

**Status of this notebook as written: never executed end-to-end (no GPU available while
building it).** Every cell was checked for Python syntax validity
(`python -m py_compile` on the extracted script) and every real API call was cross-checked
against confirmed call shapes in this repo (`qwen_vl_smoke_test.py`, `coordinate_math.py`) —
but **`qwen_vl_smoke_test.py` itself has never actually been run either** (no recorded
output found anywhere in this repo as of 2026-09-15). That means the very first cell that
calls `qwen_model.generate(...)` in this notebook is *also* the first real execution of
that call shape. Treat Section 4's first run as validating both this notebook AND the
original smoke test simultaneously — if it fails, check `qwen_vl_smoke_test.py`'s call shape
first, not just this notebook's wiring around it.

## What this notebook does

Two frozen-backbone proxy evaluations (plan §2), against all 10 real
`session_jetson_track4_*` sessions (plan §3, session-level leave-one-session-out, never
frame-level), for as many of the shortlisted candidates as actually load successfully:

- **Metric A** — zero-shot language-grounded target-marker localization hit-rate (@20mm),
  scored per session, per color command.
- **Metric B** — frozen-backbone linear-probe near-future ball-position error (mm), vision-only
  vs. vision+state, swept over a small range of `N` (frames-ahead), LOSO across all 10 sessions.

**Candidates:** Qwen2.5-VL-3B-Instruct (primary, real, already smoke-tested code path),
Jetson-PI/π0.5 (secondary, **new integration work** — not downloaded/wired anywhere in this
repo before this notebook), NVIDIA Isaac GR00T N1 (conditional — gated behind an early
precheck cell that must pass before any GR00T weights are downloaded), SmolVLA (optional,
harness-validation only, not a comparison entrant — plan §4/open item 6).

## Structure — built for "leave it running overnight," not "babysit each cell"

Every expensive stage (data staging, model download, per-session Metric A/B runs, probe
fits) is wrapped in a `run_checkpointed(stage_name, fn)` call (Section 1). A stage that
already completed in a previous run is skipped on re-run; a stage that raises is recorded as
`"failed"` with its traceback and **does not stop the notebook** — later, independent stages
still execute. Use `Runtime > Run all` and walk away; re-running `Run all` after a disconnect
resumes from wherever it left off rather than redoing completed work.

**What survives a Colab disconnect and what doesn't, explicitly:**
- `/content/...` (local Colab disk) is wiped on disconnect. Model checkpoints and the
  extracted video/telemetry data live here — cheap/idempotent to redo (re-download from HF,
  re-extract from the Drive zip), so this is intentional, not an oversight.
- Google Drive (`GDRIVE_ROOT`) is persistent. `RESULTS_PATH` (all checkpointed stage
  results) and `FEATURE_CACHE_DIR` (Metric B's extracted per-frame features — expensive,
  many forward passes) both live there specifically **because** they're too expensive to
  silently lose on a disconnect.

## Data transfer mechanism (plan §3's open "how does the data get into Colab" question)

**Chosen: zip the 10 `session_jetson_track4_*` bronze directories + supporting manifests on
the dev machine, upload the single archive to Google Drive, mount Drive in Colab.**
`docs/DATA_STORAGE.md` explicitly names Drive as a valid *occasional off-site mirror of
already-archived artifacts* (not the bulk-data case it warns against) — 10 sessions / ~666MB
total (`session_manifest.json`'s own `jetson_track4_total_video_mb: 665.7`) is exactly that
shape: a small, deliberate, one-time archive, not a live working tree or a million-file
bronze/silver directory.

**Alternative considered and NOT implemented here: `dvc pull` over the home-server Tailscale
DVC remote.** `docs/DATA_STORAGE.md`'s "Current state" section confirms these exact Track 4
sessions are *already* `dvc add`-ed and pushed (per this session's own git history —
`a41775a feat: dvc-track Track 4 Jetson bronze sessions + Qwen2.5-VL-3B checkpoint`), and
mentions a "Colab-specific Tailscale bootstrap" documented in the separate `home_server`
repo (`docs/COLAB_SETUP.md`) — this would reuse existing infrastructure rather than a second
copy path. It was **not** chosen as the primary mechanism here because: (1) that setup doc
lives in a different, private repo not readable from this session, so its exact steps
(Tailscale authkey provisioning inside a Colab runtime, MinIO credential handling) can't be
verified or reproduced here without guessing; (2) it introduces a live network/credential
dependency into an "unattended overnight" run, vs. a Drive zip that only depends on the
user's own Google account already being logged into the Colab session. If the Tailscale
path is preferred going forward, item 6+ should pull `home_server/docs/COLAB_SETUP.md`'s
real steps directly rather than this notebook guessing at them.

### Exact archive-build steps (run these on the dev machine, NOT in Colab)

```powershell
# From the repo root (C:\Users\Admin\Documents\Windows_codespace\VRI_2026)
$staging = "$env:TEMP\track4_bootstrap_staging"
New-Item -ItemType Directory -Force -Path "$staging\sessions" | Out-Null

Copy-Item "host_software\ml_jetson_vla\data_processing\session_manifest.json" "$staging\"
Copy-Item "hardware\platform_templates\ground_truth_manifest.json" "$staging\"
Copy-Item "host_software\ml_vision\core\coordinate_math.py" "$staging\"

foreach ($s in Get-ChildItem "host_software\data\01_bronze" -Directory -Filter "session_jetson_track4_*") {
    $dest = "$staging\sessions\$($s.Name)"
    New-Item -ItemType Directory -Force -Path $dest | Out-Null
    Copy-Item "$($s.FullName)\telemetry.csv" "$dest\"
    Copy-Item "$($s.FullName)\rgb_video.mp4" "$dest\"
}

Compress-Archive -Path "$staging\*" -DestinationPath "$env:TEMP\track4_bootstrap_data.zip" -Force
# Then upload $env:TEMP\track4_bootstrap_data.zip to Google Drive at:
#   MyDrive/vri2026_track4_bootstrap/track4_bootstrap_data.zip
```

Note the `01_bronze` DVC scoped exception (`docs/DATA_STORAGE.md`) means these session
directories may need `dvc pull` on the dev machine first if they aren't already materialized
locally — this step is orthogonal to (and happens before) the Colab-side transfer above.

`coordinate_math.py` is vendored into the archive verbatim (read-only copy, not modified) so
`HomographyProjector`/`PixelToPhysicalMapper` are importable in Colab without needing the
whole `ml_vision` package — reusing the real, already-calibrated homography logic rather than
reimplementing it, per `feedback_reuse_existing_export_tooling`.


## Section 1 — Environment setup and the checkpoint/resume harness

In [ ]:
#@title 1.1 Install dependencies
!pip install -q transformers accelerate qwen-vl-utils huggingface_hub \
    opencv-python-headless scikit-learn pandas numpy pillow


In [ ]:
#@title 1.2 Imports
import os
import re
import io
import sys
import gc
import json
import time
import glob
import zipfile
import traceback
from typing import Optional, Callable, Any

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch


In [ ]:
#@title 1.3 Mount Drive, define persistent vs. ephemeral paths
from google.colab import drive
drive.mount('/content/drive')

# Persistent (survives disconnects) -- see the intro markdown cell for why these three
# specifically live on Drive and nothing else does.
GDRIVE_ROOT = "/content/drive/MyDrive/vri2026_track4_bootstrap"
os.makedirs(GDRIVE_ROOT, exist_ok=True)

DRIVE_ARCHIVE_PATH = os.path.join(GDRIVE_ROOT, "track4_bootstrap_data.zip")
RESULTS_PATH = os.path.join(GDRIVE_ROOT, "bootstrap_comparison_results.json")
FEATURE_CACHE_DIR = os.path.join(GDRIVE_ROOT, "feature_cache")
os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)
COMPARISON_TABLE_CSV = os.path.join(GDRIVE_ROOT, "bootstrap_comparison_table.csv")

# Ephemeral (local Colab disk -- wiped on disconnect, cheap to regenerate)
LOCAL_DATA_ROOT = "/content/track4_data"
LOCAL_MODELS_ROOT = "/content/models"
os.makedirs(LOCAL_MODELS_ROOT, exist_ok=True)

print(f"Persistent (Drive): {GDRIVE_ROOT}")
print(f"Ephemeral (local):  {LOCAL_DATA_ROOT}, {LOCAL_MODELS_ROOT}")


In [ ]:
#@title 1.4 Checkpoint/resume harness -- the core "leave it running overnight" mechanism
def _load_results() -> dict:
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            return json.load(f)
    return {"stages": {}}


def _save_results(results: dict) -> None:
    # Write-to-temp-then-replace so a crash mid-save never corrupts the results file.
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(results, f, indent=2, default=str)
    os.replace(tmp, RESULTS_PATH)


RESULTS = _load_results()
print(f"Loaded {len(RESULTS['stages'])} previously-checkpointed stage(s) from {RESULTS_PATH}")


def run_checkpointed(stage_name: str, fn: Callable[[], Any], force: bool = False) -> Any:
    """Runs fn() at most once per stage_name across the life of this results file.

    - Already-'complete' stages are skipped on re-run (idempotent 'Run all'), unless
      force=True.
    - A stage that raises is recorded as 'failed' with its traceback and this function
      returns None WITHOUT re-raising -- later, independent run_checkpointed() calls in
      subsequent cells still execute. This is what makes an unattended overnight run
      survive one candidate's failure (e.g. an unconfirmed OpenPI API call) without
      losing every other stage that would otherwise have completed after it.
    - Every call persists RESULTS to Drive immediately, so a Colab disconnect loses at
      most the one in-flight stage, never previously-completed ones.
    """
    entry = RESULTS["stages"].get(stage_name)
    if entry is not None and entry.get("status") == "complete" and not force:
        print(f"[SKIP] {stage_name} (already complete)")
        return entry.get("result")

    print(f"[RUN ] {stage_name} ...")
    t0 = time.time()
    try:
        result = fn()
        RESULTS["stages"][stage_name] = {
            "status": "complete",
            "result": result,
            "elapsed_sec": round(time.time() - t0, 2),
            "finished_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        print(f"[DONE] {stage_name} ({time.time() - t0:.1f}s)")
    except Exception as e:  # noqa: BLE001 -- intentionally broad, see docstring
        RESULTS["stages"][stage_name] = {
            "status": "failed",
            "error": str(e),
            "traceback": traceback.format_exc(),
            "elapsed_sec": round(time.time() - t0, 2),
            "finished_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        print(f"[FAIL] {stage_name}: {e}")

    _save_results(RESULTS)
    return RESULTS["stages"][stage_name].get("result")


def stage_status(stage_name: str) -> Optional[str]:
    entry = RESULTS["stages"].get(stage_name)
    return entry.get("status") if entry else None


## Section 2 — Data staging

Extracts the archive built by the dev-machine PowerShell steps above (must already be
uploaded to `DRIVE_ARCHIVE_PATH`). Extraction target is local disk (`LOCAL_DATA_ROOT`) for
fast video decode during feature extraction -- cheap to redo from the persistent Drive zip
after any disconnect.

In [ ]:
#@title 2.1 Extract session archive
def _stage_data_transfer():
    if not os.path.exists(DRIVE_ARCHIVE_PATH):
        raise FileNotFoundError(
            f"{DRIVE_ARCHIVE_PATH} not found. Build it on the dev machine using the "
            f"PowerShell steps in the intro markdown cell, then upload it to that exact "
            f"Drive path before re-running this cell."
        )
    os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ARCHIVE_PATH) as zf:
        zf.extractall(LOCAL_DATA_ROOT)
    return {"extracted_to": LOCAL_DATA_ROOT, "archive_size_bytes": os.path.getsize(DRIVE_ARCHIVE_PATH)}


run_checkpointed("data_transfer_extract_archive", _stage_data_transfer)


In [ ]:
#@title 2.2 Load manifests, vendor coordinate_math.py, sanity-check session count
sys.path.insert(0, LOCAL_DATA_ROOT)
from coordinate_math import HomographyProjector  # noqa: E402  (vendored, read-only copy)

with open(os.path.join(LOCAL_DATA_ROOT, "ground_truth_manifest.json")) as f:
    GT_MANIFEST = json.load(f)

with open(os.path.join(LOCAL_DATA_ROOT, "session_manifest.json")) as f:
    SESSION_MANIFEST = json.load(f)

TRACK4_SESSIONS = [s for s in SESSION_MANIFEST["sessions"] if s["regime"] == "jetson_track4_rl_teacher"]
assert len(TRACK4_SESSIONS) == 10, f"expected 10 Track 4 sessions per the plan, found {len(TRACK4_SESSIONS)}"
print(f"Loaded {len(TRACK4_SESSIONS)} Track 4 sessions:")
for s in TRACK4_SESSIONS:
    print(f"  - {os.path.basename(s['session_dir'])}  ({s['video_frame_count']} frames)")

# Corner markers, in the order ground_truth_manifest.json lists them (top-left, top-right,
# bottom-right, bottom-left) -- matches HomographyProjector's expected src/dst point order.
CORNER_MARKER_IDS = [0, 1, 2, 3]
_marker_by_id = {m["id"]: m for m in GT_MANIFEST["aruco_markers"]}
DST_PTS_MM = np.array([_marker_by_id[i]["center_mm"] for i in CORNER_MARKER_IDS], dtype=np.float32)
print(f"Platform corner markers (mm, dst points for homography): {DST_PTS_MM.tolist()}")


**Note on Metric A's ground-truth/scoring direction (a clarification made while
implementing the plan, not a change to it):** `coordinate_math.py`'s real
`HomographyProjector.project_point(px, py)` only projects **pixel -> mm** (`cv2.perspectiveTransform`
with the fitted forward matrix); there is no inverse (`mm -> px`) method on the class. Plan
§2.1's "Ground truth ... converted to pixel space" phrasing and its "Scoring" bullet's
"converted back to mm" phrasing are subtly inconsistent about which direction the conversion
runs. This notebook implements the direction the real class actually supports: the
**candidate's predicted pixel point** is projected to mm via `project_point()`, then compared
directly against the session's real `target_x`/`target_y` (already in mm in `telemetry.csv`,
no conversion needed). Net effect is the same comparison the plan intends (predicted vs. true
target location, in a common mm space, 20mm tolerance) -- just resolved against the real,
one-directional API rather than an inverse that doesn't exist.

In [ ]:
#@title 2.3 Per-session homography (ArUco corner-marker detection -> HomographyProjector)
def compute_session_homography(video_path: str, dict_type=cv2.aruco.DICT_4X4_50, max_frames_to_try: int = 60):
    """Detects the 4 corner ArUco markers (ids 0-3) on an early frame of the session and
    fits a pixel->mm homography via the real HomographyProjector. Assumes a static camera
    for the whole session (fixed rig, matches how this platform is actually operated) --
    computed once per session, not once per frame, to keep this cheap."""
    try:
        aruco_dict = cv2.aruco.getPredefinedDictionary(dict_type)
        aruco_params = cv2.aruco.DetectorParameters()
        detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)
        def _detect(gray):
            corners, ids, _ = detector.detectMarkers(gray)
            return corners, ids
    except AttributeError:
        aruco_dict = cv2.aruco.Dictionary_get(dict_type)
        aruco_params = cv2.aruco.DetectorParameters_create()
        def _detect(gray):
            corners, ids, _ = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=aruco_params)
            return corners, ids

    cap = cv2.VideoCapture(video_path)
    try:
        for _ in range(max_frames_to_try):
            ret, frame = cap.read()
            if not ret:
                break
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            corners, ids = _detect(gray)
            if ids is None:
                continue
            ids_flat = ids.flatten().tolist()
            if all(m_id in ids_flat for m_id in CORNER_MARKER_IDS):
                centers_px = []
                for m_id in CORNER_MARKER_IDS:
                    idx = ids_flat.index(m_id)
                    corner_pts = corners[idx][0]  # (4, 2): the marker's own 4 corners
                    centers_px.append(corner_pts.mean(axis=0))
                projector = HomographyProjector(DST_PTS_MM)
                ok = projector.update_homography(np.array(centers_px, dtype=np.float32))
                if ok:
                    return projector
    finally:
        cap.release()
    return None


def _stage_compute_homographies():
    out = {}
    for s in TRACK4_SESSIONS:
        session_short = os.path.basename(s["session_dir"])
        video_path = os.path.join(LOCAL_DATA_ROOT, "sessions", session_short, "rgb_video.mp4")
        projector = compute_session_homography(video_path)
        out[session_short] = {"found": projector is not None}
        if projector is not None:
            out[session_short]["homography_matrix"] = projector.M.tolist()
    n_found = sum(1 for v in out.values() if v["found"])
    print(f"Homography found for {n_found}/{len(out)} sessions")
    return out


HOMOGRAPHY_RESULT = run_checkpointed("data_compute_session_homographies", _stage_compute_homographies)

# Rebuild live HomographyProjector objects from the cached matrices (numpy arrays aren't
# JSON-native, so the checkpointed result stores lists -- reconstruct actual projectors here
# for use in later cells, cheap to redo every runtime start).
SESSION_HOMOGRAPHIES = {}
if HOMOGRAPHY_RESULT:
    for session_short, info in HOMOGRAPHY_RESULT.items():
        if info.get("found"):
            proj = HomographyProjector(DST_PTS_MM)
            proj.M = np.array(info["homography_matrix"], dtype=np.float64)
            SESSION_HOMOGRAPHIES[session_short] = proj


## Section 3 — Shared config and Metric A/B utilities (candidate-agnostic)

In [ ]:
#@title 3.1 Config knobs (tune these down first if a run is too slow for one overnight session)
TOLERANCE_MM = 20.0  # Metric A hit tolerance -- reused from EVALUATION_STRATEGY.md's settling-time radius, per plan §2.1
COLOR_COMMANDS = ["go_red", "go_green", "go_yellow", "go_black"]
COLOR_COMMAND_TO_WORD = {"go_red": "red", "go_green": "green", "go_yellow": "yellow", "go_black": "black"}

# Metric A: process every Nth eligible color-command frame per session (compute-budget knob;
# lower = more coverage = slower). ~2000 color-command frames exist across all 10 sessions
# combined at stride 1 -- one generate() call per Qwen frame is the dominant cost here.
FRAME_SAMPLE_STRIDE_METRIC_A = 5

# Metric B: subsample frames for feature extraction (compute-budget knob).
FRAME_SAMPLE_STRIDE_METRIC_B = 3

# Plan §2.2: N starting point 10 (333ms @ 30fps), sweep ~5-15 rather than commit to one value.
N_FUTURE_SWEEP = [5, 7, 10, 13, 15]

REQUIRED_TELEMETRY_COLUMNS = [
    "frame_index", "host_timestamp_ms", "target_x", "target_y",
    "touch_x", "touch_y", "theta_a", "theta_b", "theta_c",
]


In [ ]:
#@title 3.2 Metric A: prompt template + best-effort point parser (UNVERIFIED, see note)
LOCALIZATION_PROMPT_TEMPLATE = (
    "The robot must move the ball to the {color} marker. "
    "Point to the marker's location in this image. "
    "Respond with only the pixel coordinates as (x, y)."
)

# UNVERIFIED (plan §5 open item 1, restated in this notebook's intro): Qwen2.5-VL is
# documented upstream to support a grounded point/bbox output format, but that format was
# NOT confirmed against this local checkpoint + transformers version this session (no GPU
# run happened while building this notebook). This regex is a best-effort parser for a
# plain "(x, y)" text response -- it is NOT a parser for Qwen's special grounding tokens
# (e.g. <|box_start|>/<|point_start|>-style output), which this checkpoint may actually
# emit instead. If Section 4's first real run shows raw_text containing special tokens
# instead of plain numbers, THIS PARSER MUST BE REWRITTEN before any Metric A number here
# can be trusted -- Section 4 logs raw_text for every scored frame specifically so this is
# checkable after the first real run, not assumed correct in advance.
_POINT_RE = re.compile(r"\(?\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\)?")


def parse_point_from_text(text: str):
    m = _POINT_RE.search(text or "")
    if not m:
        return None
    return float(m.group(1)), float(m.group(2))


In [ ]:
#@title 3.3 Metric A: generic per-session scoring loop (candidate-agnostic; takes a generate_fn)
def metric_a_score_session(session_meta: dict, generate_fn: Callable[[np.ndarray, str], str],
                            candidate_name: str, stride: int = FRAME_SAMPLE_STRIDE_METRIC_A) -> dict:
    """generate_fn(frame_bgr, prompt_text) -> raw text response, in the candidate's own
    call shape. Scores against the per-session homography computed in Section 2."""
    session_short = os.path.basename(session_meta["session_dir"])
    session_dir = os.path.join(LOCAL_DATA_ROOT, "sessions", session_short)
    csv_path = os.path.join(session_dir, "telemetry.csv")
    video_path = os.path.join(session_dir, "rgb_video.mp4")

    projector = SESSION_HOMOGRAPHIES.get(session_short)
    if projector is None:
        return {"session": session_short, "error": "no_homography_available", "candidate": candidate_name}

    df = pd.read_csv(csv_path)
    cap = cv2.VideoCapture(video_path)
    per_color_hits = {c: 0 for c in COLOR_COMMANDS}
    per_color_total = {c: 0 for c in COLOR_COMMANDS}
    frame_records = []
    seen_color_frames = 0
    try:
        for _, row in df.iterrows():
            ret, frame = cap.read()
            if not ret:
                break
            cmd = str(row.get("audio_command", "")).strip()
            if cmd not in COLOR_COMMANDS:
                continue
            seen_color_frames += 1
            if seen_color_frames % stride != 0:
                continue
            if pd.isna(row["target_x"]) or pd.isna(row["target_y"]):
                continue

            prompt = LOCALIZATION_PROMPT_TEMPLATE.format(color=COLOR_COMMAND_TO_WORD[cmd])
            raw_text = generate_fn(frame, prompt)
            point_px = parse_point_from_text(raw_text)

            per_color_total[cmd] += 1
            hit = False
            pred_mm = None
            if point_px is not None:
                mm_x, mm_y = projector.project_point(*point_px)
                if mm_x is not None:
                    pred_mm = (mm_x, mm_y)
                    dist_mm = ((mm_x - row["target_x"]) ** 2 + (mm_y - row["target_y"]) ** 2) ** 0.5
                    hit = dist_mm <= TOLERANCE_MM
                    if hit:
                        per_color_hits[cmd] += 1

            # Cap stored per-frame detail to keep the results JSON bounded on long sessions.
            if len(frame_records) < 500:
                frame_records.append({
                    "frame_index": int(row["frame_index"]), "command": cmd,
                    "raw_text": raw_text, "point_px": point_px, "pred_mm": pred_mm,
                    "target_mm": [float(row["target_x"]), float(row["target_y"])], "hit": hit,
                })
    finally:
        cap.release()

    hit_rate_per_color = {
        c: (per_color_hits[c] / per_color_total[c] if per_color_total[c] > 0 else None)
        for c in COLOR_COMMANDS
    }
    return {
        "candidate": candidate_name, "session": session_short,
        "per_color_total": per_color_total, "per_color_hits": per_color_hits,
        "hit_rate_per_color": hit_rate_per_color,
        "n_frames_scored": sum(per_color_total.values()),
        "frame_records_sample": frame_records,
    }


In [ ]:
#@title 3.4 Metric B: generic per-session feature extraction + caching (candidate-agnostic)
def extract_and_cache_features(session_meta: dict, candidate_name: str,
                                extract_fn: Callable[[np.ndarray], np.ndarray],
                                stride: int = FRAME_SAMPLE_STRIDE_METRIC_B) -> dict:
    """extract_fn(frame_bgr) -> fixed-length 1D np.ndarray pooled feature. Caches to
    FEATURE_CACHE_DIR (Drive-persistent) as one .npz per (candidate, session) -- the
    expensive part (many forward passes) survives a disconnect even if the in-memory model
    doesn't."""
    session_short = os.path.basename(session_meta["session_dir"])
    out_path = os.path.join(FEATURE_CACHE_DIR, f"{candidate_name}__{session_short}.npz")
    if os.path.exists(out_path):
        cached = np.load(out_path)
        return {"out_path": out_path, "n_frames": len(cached["frame_idx"]),
                "feature_dim": cached["features"].shape[1] if cached["features"].size else 0,
                "reused_cache": True}

    session_dir = os.path.join(LOCAL_DATA_ROOT, "sessions", session_short)
    csv_path = os.path.join(session_dir, "telemetry.csv")
    video_path = os.path.join(session_dir, "rgb_video.mp4")
    df = pd.read_csv(csv_path)
    cap = cv2.VideoCapture(video_path)
    feats, frame_idxs, touch_xy = [], [], []
    try:
        for i, row in df.iterrows():
            ret, frame = cap.read()
            if not ret:
                break
            if i % stride != 0:
                continue
            if pd.isna(row["touch_x"]) or pd.isna(row["touch_y"]):
                continue
            feats.append(extract_fn(frame))
            frame_idxs.append(int(row["frame_index"]))
            touch_xy.append([float(row["touch_x"]), float(row["touch_y"])])
    finally:
        cap.release()

    feats_arr = np.stack(feats) if feats else np.zeros((0, 1), dtype=np.float32)
    np.savez_compressed(out_path, features=feats_arr,
                         frame_idx=np.array(frame_idxs, dtype=np.int64),
                         touch_xy=np.array(touch_xy, dtype=np.float32))
    return {"out_path": out_path, "n_frames": len(frame_idxs),
            "feature_dim": feats_arr.shape[1] if feats_arr.size else 0, "reused_cache": False}


In [ ]:
#@title 3.5 Metric B: linear-probe LOSO fit/eval (candidate-agnostic, reads cached features)
from sklearn.linear_model import Ridge


def _load_cached_features(candidate_name: str, session_meta: dict):
    session_short = os.path.basename(session_meta["session_dir"])
    path = os.path.join(FEATURE_CACHE_DIR, f"{candidate_name}__{session_short}.npz")
    data = np.load(path)
    return data["features"], data["frame_idx"], data["touch_xy"]


def _build_probe_examples(features: np.ndarray, frame_idx: np.ndarray, touch_xy: np.ndarray, n_future: int):
    """Pairs feature[i] (+ current-state touch_xy[i]) with the touch_xy of the nearest
    CACHED sample whose frame_idx >= frame_idx[i] + n_future. Frames were subsampled at
    extraction time (FRAME_SAMPLE_STRIDE_METRIC_B), so exact +n_future alignment isn't
    guaranteed -- nearest-available-at-or-after is the practical fallback; flagged here
    rather than silently assumed exact, since it means effective lookahead is
    n_future..n_future+stride-1 frames, not exactly n_future."""
    X_vision, X_state, y = [], [], []
    for i in range(len(frame_idx)):
        target_frame = frame_idx[i] + n_future
        future_pos = int(np.searchsorted(frame_idx, target_frame))
        if future_pos >= len(frame_idx):
            continue
        X_vision.append(features[i])
        X_state.append(touch_xy[i])
        y.append(touch_xy[future_pos])
    if not y:
        return np.zeros((0, features.shape[1] if features.size else 1)), np.zeros((0, 2)), np.zeros((0, 2))
    return np.array(X_vision), np.array(X_state), np.array(y)


def metric_b_probe_loso(candidate_name: str, n_future: int, variant: str) -> dict:
    """variant: 'vision_only' or 'vision_plus_state'. LOSO across all 10 Track4 sessions,
    per plan §3 (session-level splitting is load-bearing, never frame-level)."""
    assert variant in ("vision_only", "vision_plus_state")

    examples = {}
    for s in TRACK4_SESSIONS:
        feats, fidx, txy = _load_cached_features(candidate_name, s)
        examples[s["session_dir"]] = _build_probe_examples(feats, fidx, txy, n_future)

    fold_results = []
    for held_out in TRACK4_SESSIONS:
        ho_key = held_out["session_dir"]
        train_keys = [s["session_dir"] for s in TRACK4_SESSIONS if s["session_dir"] != ho_key]

        def _stack(keys):
            xv, xs, y = [], [], []
            for k in keys:
                a, b, c = examples[k]
                if len(c) == 0:
                    continue
                xv.append(a); xs.append(b); y.append(c)
            if not y:
                return None, None, None
            return np.concatenate(xv), np.concatenate(xs), np.concatenate(y)

        Xv_tr, Xs_tr, y_tr = _stack(train_keys)
        Xv_te, Xs_te, y_te = examples[ho_key]

        if Xv_tr is None or len(y_te) == 0:
            fold_results.append({"held_out": os.path.basename(ho_key), "error": "insufficient_examples"})
            continue

        if variant == "vision_only":
            X_tr, X_te = Xv_tr, Xv_te
        else:
            X_tr = np.concatenate([Xv_tr, Xs_tr], axis=1)
            X_te = np.concatenate([Xv_te, Xs_te], axis=1)

        model = Ridge(alpha=1.0)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        err_mm = np.linalg.norm(pred - y_te, axis=1)
        fold_results.append({
            "held_out": os.path.basename(ho_key), "n_train": int(len(y_tr)), "n_eval": int(len(y_te)),
            "mean_error_mm": float(err_mm.mean()), "std_error_mm": float(err_mm.std()),
        })

    valid = [f for f in fold_results if "mean_error_mm" in f]
    aggregate = {
        "n_folds_valid": len(valid),
        "mean_error_mm_across_folds": float(np.mean([f["mean_error_mm"] for f in valid])) if valid else None,
        "std_error_mm_across_folds": float(np.std([f["mean_error_mm"] for f in valid])) if valid else None,
    }
    return {"candidate": candidate_name, "n_future": n_future, "variant": variant,
            "folds": fold_results, "aggregate": aggregate}


## Section 4 — Candidate A: Qwen2.5-VL-3B-Instruct (primary)

Zero new download cost per the plan (already staged on the dev machine) -- but for Colab
this re-downloads from `Qwen/Qwen2.5-VL-3B-Instruct` on Hugging Face rather than assuming the
dev machine's local `models/qwen2_5_vl_3b_instruct/` path is reachable from a Colab runtime
(it isn't). The `from_pretrained`/`generate()` call shape below is copied verbatim from
`deployment/qwen_vl_smoke_test.py` -- the one real, existing reference for this model's API
in this repo -- **not re-derived**, per `feedback_reuse_existing_export_tooling`.

In [ ]:
#@title 4.1 Download + load Qwen2.5-VL-3B-Instruct
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

QWEN_HF_REPO = "Qwen/Qwen2.5-VL-3B-Instruct"
QWEN_LOCAL_DIR = os.path.join(LOCAL_MODELS_ROOT, "qwen2_5_vl_3b_instruct")


def _stage_download_qwen():
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=QWEN_HF_REPO, local_dir=QWEN_LOCAL_DIR)
    return {"local_dir": QWEN_LOCAL_DIR}


run_checkpointed("qwen_download", _stage_download_qwen)

# NOT wrapped in run_checkpointed: loading weights into GPU memory can't be "checkpointed"
# across a disconnect (a fresh runtime always needs to reload), only the download above is
# worth persisting/skipping. This is also, per the intro markdown, the first real execution
# of this exact call shape -- qwen_vl_smoke_test.py has never been run.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen2.5-VL-3B-Instruct on {device} ...")
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_LOCAL_DIR, torch_dtype=torch.float16, device_map="auto",
)
qwen_processor = AutoProcessor.from_pretrained(QWEN_LOCAL_DIR)
print("Qwen2.5-VL-3B-Instruct loaded.")


In [ ]:
#@title 4.2 Qwen generate() wrapper -- matches qwen_vl_smoke_test.py's exact call shape
_QWEN_TMP_IMG = "/content/_qwen_tmp_frame.jpg"


def qwen_generate(frame_bgr: np.ndarray, prompt_text: str, max_new_tokens: int = 64) -> str:
    # Written to a temp file and passed as a path, exactly like qwen_vl_smoke_test.py's
    # image_path argument -- deliberately NOT passing a PIL.Image object directly, since
    # that variant of qwen_vl_utils' accepted input types was not confirmed this session
    # and the file-path form is the only one with a real (if unexecuted) reference.
    cv2.imwrite(_QWEN_TMP_IMG, frame_bgr)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": _QWEN_TMP_IMG, "max_pixels": 1280 * 28 * 28},
            {"type": "text", "text": prompt_text},
        ],
    }]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(text=[text], images=image_inputs, videos=video_inputs,
                             padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    return qwen_processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]


In [ ]:
#@title 4.3 Metric A for Qwen -- one checkpointed stage per session
for _session in TRACK4_SESSIONS:
    _short = os.path.basename(_session["session_dir"])
    run_checkpointed(
        f"metric_a_qwen__{_short}",
        lambda sm=_session: metric_a_score_session(sm, qwen_generate, "qwen2.5-vl-3b"),
    )


In [ ]:
#@title 4.4 Qwen vision-tower/merger feature extraction for Metric B (UNVERIFIED API, see note)
def qwen_extract_vision_feature(frame_bgr: np.ndarray) -> np.ndarray:
    """Pools Qwen2.5-VL's vision-tower+merger output (2048-dim, per
    MULTI_HEAD_ARCHITECTURE_SPEC.md Section 1's real config-dump numbers) -- the plan's
    'vision-language backbone up to a fixed pooling point ... Qwen's merger output'
    extraction point (plan Section 2.2), NOT a full 36-layer LM decoder forward pass
    (cheaper, and matches the plan's wording literally).

    UNVERIFIED: `model.visual(pixel_values, grid_thw=...)` as the exact submodule
    name/call signature for Qwen2_5_VLForConditionalGeneration was read from this repo's
    own architecture-spec discussion of the checkpoint's config.json, NOT executed against
    the installed transformers version this session (no GPU available). Confirm this
    attribute/signature on first real run -- if `model.visual` doesn't exist or has a
    different signature in the installed transformers version, this is the first thing to
    fix before trusting any Qwen Metric B number.
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    messages = [{"role": "user", "content": [{"type": "image", "image": pil_image, "max_pixels": 1280 * 28 * 28}]}]
    image_inputs, _ = process_vision_info(messages)
    inputs = qwen_processor(text=[""], images=image_inputs, videos=None, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        image_embeds = qwen_model.visual(inputs["pixel_values"], grid_thw=inputs["image_grid_thw"])
    return image_embeds.mean(dim=0).float().cpu().numpy()  # (2048,)


for _session in TRACK4_SESSIONS:
    _short = os.path.basename(_session["session_dir"])
    run_checkpointed(
        f"metric_b_features_qwen__{_short}",
        lambda sm=_session: extract_and_cache_features(sm, "qwen2.5-vl-3b", qwen_extract_vision_feature),
    )


In [ ]:
#@title 4.5 Metric B probe fit/eval for Qwen -- swept over N and both state variants
for _n_future in N_FUTURE_SWEEP:
    for _variant in ("vision_only", "vision_plus_state"):
        _stage = f"metric_b_probe_qwen2.5-vl-3b__n{_n_future}__{_variant}"
        run_checkpointed(_stage, lambda nf=_n_future, v=_variant: metric_b_probe_loso("qwen2.5-vl-3b", nf, v))


In [ ]:
#@title 4.6 Free Qwen's GPU memory before the next candidate
del qwen_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Qwen2.5-VL-3B unloaded.")


## Section 5 — NVIDIA Isaac GR00T N1 precheck gate (conditional candidate)

Per plan §1: GR00T N1 is **not** downloaded blind. This cell fetches only `config.json` +
license metadata (no weights) for candidate repo ids and lets a human decide `eligible`
before Section 6 runs at all. **The exact smallest-variant repo id was not confirmed this
session** (plan §5 open item 3) -- the ids below are best-guess from NVIDIA's public
`Isaac-GR00T` naming, not a verified listing. `eligible` defaults to `False`; a human must
read `groot_gate_result["findings"]` and flip it (or extend this cell's logic once the real
repo id + param count are confirmed) before Section 6's cells will do anything.

In [ ]:
#@title 5.1 GR00T N1 precheck (config/license only, no weights) -- GATES Section 6
GROOT_HF_REPO_CANDIDATES = ["nvidia/GR00T-N1-2B", "nvidia/GR00T-N1.5-3B"]  # UNCONFIRMED, see markdown above
GROOT_PARAM_CEILING = 4e9


def _groot_precheck():
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    findings = {}
    for repo_id in GROOT_HF_REPO_CANDIDATES:
        try:
            info = api.model_info(repo_id)
            license_tag = next((t for t in (info.tags or []) if t.startswith("license:")), "unknown")
            config_path = hf_hub_download(repo_id=repo_id, filename="config.json")
            with open(config_path) as f:
                cfg = json.load(f)
            findings[repo_id] = {"license": license_tag, "config_keys_present": list(cfg.keys())[:20]}
        except Exception as e:  # noqa: BLE001 -- a missing/wrong repo id must not kill the notebook
            findings[repo_id] = {"error": str(e)}
    return {
        "eligible": False,  # deliberately conservative default -- see note below
        "findings": findings,
        "note": (
            "eligible defaults to False. Read `findings` (per-repo license tag + config "
            "keys) and set GROOT_ELIGIBLE = True manually in the next cell only after "
            "confirming a sub-4B-param variant with an acceptable license actually exists "
            "at one of these repo ids -- this cell does not compute an exact param count "
            "from config.json alone, that still needs a human judgment call per plan Section 1."
        ),
    }


groot_gate_result = run_checkpointed("groot_n1_precheck_gate", _groot_precheck)

# Human-set flag -- flip to True only after reading groot_gate_result["findings"] above and
# confirming a sub-ceiling, acceptably-licensed variant. Left False here deliberately.
GROOT_ELIGIBLE = False
print(f"GR00T N1 gate: eligible={GROOT_ELIGIBLE} -- Section 6 cells will "
      f"{'run' if GROOT_ELIGIBLE else 'be skipped'}.")


## Section 6 — Candidate C: NVIDIA Isaac GR00T N1 (only runs if Section 5's gate passed)

In [ ]:
#@title 6.1 GR00T N1 download + load (only if GROOT_ELIGIBLE)
def _stage_groot_skip():
    return {"skipped": True, "reason": "groot_n1_precheck_gate did not set GROOT_ELIGIBLE=True"}


if GROOT_ELIGIBLE:
    def _stage_download_groot():
        # UNVERIFIED: exact loading API for whichever GR00T-N1 repo the gate confirmed --
        # not designed further here since the gate has not passed as of writing this
        # notebook. Fill in the real snapshot_download repo_id + model class once Section
        # 5 confirms a specific eligible variant.
        raise NotImplementedError(
            "GR00T N1 gate passed but this cell's load logic was never filled in against "
            "a confirmed repo id -- do that before relying on this candidate."
        )
    run_checkpointed("groot_download_load", _stage_download_groot)
else:
    run_checkpointed("groot_download_load", _stage_groot_skip)


In [ ]:
#@title 6.2 GR00T N1 Metric A + Metric B (only if GROOT_ELIGIBLE)
if GROOT_ELIGIBLE:
    raise NotImplementedError(
        "Fill in GR00T-specific generate_fn / extract_fn here once Section 6.1's load "
        "logic is real, then reuse metric_a_score_session / extract_and_cache_features / "
        "metric_b_probe_loso exactly like Section 4 does for Qwen."
    )
else:
    print("GR00T N1 gate not passed -- skipping Metric A/B for this candidate.")
    run_checkpointed("metric_a_groot_all_sessions", _stage_groot_skip)
    run_checkpointed("metric_b_probe_groot_all", _stage_groot_skip)


## Section 7 — Candidate B: Jetson-PI / π0.5 (OpenPI) — secondary, real new integration work

Unlike Qwen, **nothing about this candidate exists in this repo yet** -- no checkpoint, no
loading code, no prior smoke test. This section is genuinely new integration work, as the
plan (§4) says it should be. Every API call below beyond the plain `git clone` is
**unverified** against the real `openpi`/`Jetson-PI` source (no local checkpoint existed to
test against while building this notebook, unlike Qwen's `qwen_vl_smoke_test.py`) -- each
stage is still wrapped in `run_checkpointed` so a wrong guess here fails loudly, gets
recorded, and does **not** stop Section 4/5/6/8 (already-run or independent) from
completing.

In [ ]:
#@title 7.1 Clone + install openpi
def _stage_clone_openpi():
    import subprocess
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Physical-Intelligence/openpi.git",
         "/content/openpi"], check=True,
    )
    subprocess.run(["pip", "install", "-q", "-e", "/content/openpi"], check=True)
    return {"cloned_to": "/content/openpi"}


run_checkpointed("pi05_clone_install_openpi", _stage_clone_openpi)


In [ ]:
#@title 7.2 Load a released pi0.5 checkpoint (UNVERIFIED call shape -- see 7.0 markdown)
def _stage_load_pi05():
    # Best-effort against openpi's documented pattern (policy_config + a named train
    # config pointing at a released checkpoint) -- the exact config name and checkpoint
    # location were NOT confirmed against the real repo this session (LARGE_VLA_RESEARCH_
    # SPIKE.md names the GitHub repos, not a specific checkpoint identifier/config name).
    # Read /content/openpi's own README/examples after 7.1's clone completes and correct
    # this before trusting a "complete" status here.
    sys.path.insert(0, "/content/openpi/src")
    from openpi.policies import policy_config  # noqa: E402
    from openpi.training import config as openpi_train_config  # noqa: E402

    cfg = openpi_train_config.get_config("pi05_base")  # PLACEHOLDER config name, unconfirmed
    policy = policy_config.create_trained_policy(cfg, checkpoint_dir=None)  # PLACEHOLDER
    return {"loaded": True}


pi05_load_result = run_checkpointed("pi05_load_checkpoint", _stage_load_pi05)
PI05_AVAILABLE = stage_status("pi05_load_checkpoint") == "complete"
print(f"pi0.5 available: {PI05_AVAILABLE}")
if not PI05_AVAILABLE:
    print("pi0.5 did not load (expected on a first real run against unconfirmed API calls "
          "-- see RESULTS['stages']['pi05_load_checkpoint']['traceback'] once this actually "
          "executes). Sections 7.3/7.4 below record an honest 'skipped' result rather than "
          "fabricating numbers.")


In [ ]:
#@title 7.3 Metric A for pi0.5 (only if it exposes a text-grounded localization path)
def _stage_pi05_metric_a_skip():
    return {"skipped": True, "reason": (
        "pi05 not loaded, OR pi0.5/PaliGemma's architecture may not expose the same "
        "plain-text 'point to the marker' generation path Qwen does -- pi0.5 is primarily "
        "an action-expert (flow-matching) model, not a general VLM chat interface. This "
        "was NOT checked against a real loaded policy this session; Metric A may turn out "
        "to be not applicable to this candidate at all, which is an honest possible "
        "outcome, not a bug to work around."
    )}


if PI05_AVAILABLE:
    for _session in TRACK4_SESSIONS:
        _short = os.path.basename(_session["session_dir"])
        run_checkpointed(f"metric_a_pi05__{_short}", _stage_pi05_metric_a_skip)  # replace once 7.2's load is real and a grounding path is confirmed
else:
    run_checkpointed("metric_a_pi05_all_sessions", _stage_pi05_metric_a_skip)


In [ ]:
#@title 7.4 Metric B feature extraction for pi0.5 (PaliGemma embedding pooling point, UNVERIFIED)
def _stage_pi05_metric_b_skip():
    return {"skipped": True, "reason": (
        "pi05 not loaded, OR whether pi0.5's architecture cleanly exposes a poolable "
        "frozen hidden state the way Qwen's merger output does was never checked (plan "
        "Section 5 open item 2) -- if it doesn't cleanly separate from the action-expert "
        "path, Metric B needs a different extraction point for this candidate specifically, "
        "not attempted here without a loaded policy to inspect."
    )}


if PI05_AVAILABLE:
    for _session in TRACK4_SESSIONS:
        _short = os.path.basename(_session["session_dir"])
        run_checkpointed(f"metric_b_features_pi05__{_short}", _stage_pi05_metric_b_skip)  # replace with a real extract_fn once 7.2 loads for real
    # Plan Section 2.2's third variant for this candidate specifically: its own native
    # flow-matching action-expert's predicted next-state, scored the same way. Also
    # unimplemented pending a real loaded policy -- placeholder stage recorded honestly.
    run_checkpointed("metric_b_pi05_native_action_expert_variant", _stage_pi05_metric_b_skip)
else:
    run_checkpointed("metric_b_features_pi05_all_sessions", _stage_pi05_metric_b_skip)


## Section 8 — SmolVLA harness-validation (optional, NOT a comparison entrant)

Plan §4 / open item 6: SmolVLA (450M params) is explicitly **not** a shortlist candidate --
it's included only, optionally, as a cheap way to validate this notebook's probe-fitting/
prompt-parsing harness before spending expensive time on Qwen/π0.5. Since Section 4 (Qwen)
above is already a zero-download-cost real candidate, this section is OFF by default
(`RUN_SMOLVLA_HARNESS_CHECK = False`) -- flip it on only if you specifically want an extra,
even-cheaper sanity check of the harness plumbing itself, independent of any real candidate's
numbers. This mirrors the plan's own framing of it as optional, not required.

In [ ]:
#@title 8.1 SmolVLA harness smoke test (off by default)
RUN_SMOLVLA_HARNESS_CHECK = False

if RUN_SMOLVLA_HARNESS_CHECK:
    def _stage_smolvla_harness_check():
        # Deliberately minimal -- this only needs to prove metric_a_score_session /
        # extract_and_cache_features / metric_b_probe_loso run end-to-end against SOME
        # loaded model, not produce a scored comparison entrant. Fill in a real SmolVLA
        # load (e.g. via `lerobot`'s policy loading, already a project dependency per
        # convert_to_lerobot.py) before enabling this.
        raise NotImplementedError("Fill in a minimal SmolVLA load here if this harness check is enabled.")
    run_checkpointed("smolvla_harness_check", _stage_smolvla_harness_check)
else:
    print("SmolVLA harness check skipped (RUN_SMOLVLA_HARNESS_CHECK=False, the default -- see markdown above).")


## Section 9 — Final aggregation

Reads whatever is in `RESULTS` right now -- runs correctly whether every section above
finished or the notebook is being re-run mid-way through an overnight session. Reports
completed vs. failed/skipped stages explicitly rather than silently omitting them.

In [ ]:
#@title 9.1 Build and save the comparison table
def build_comparison_table() -> pd.DataFrame:
    rows = []
    for stage_name, entry in RESULTS["stages"].items():
        if entry.get("status") != "complete":
            continue
        res = entry.get("result")
        if not isinstance(res, dict):
            continue
        if res.get("skipped"):
            continue
        if "aggregate" in res and "candidate" in res and "variant" in res:
            agg = res["aggregate"]
            rows.append({
                "stage": stage_name, "metric": "B", "candidate": res["candidate"],
                "variant": res.get("variant"), "n_future": res.get("n_future"),
                "mean_error_mm": agg.get("mean_error_mm_across_folds"),
                "std_error_mm": agg.get("std_error_mm_across_folds"),
                "n_folds_valid": agg.get("n_folds_valid"),
            })
        elif "hit_rate_per_color" in res:
            row = {
                "stage": stage_name, "metric": "A", "candidate": res.get("candidate"),
                "session": res.get("session"), "n_frames_scored": res.get("n_frames_scored"),
            }
            row.update({f"hit_rate_{c}": v for c, v in res["hit_rate_per_color"].items()})
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(COMPARISON_TABLE_CSV, index=False)
    print(f"Saved comparison table ({len(df)} rows) -> {COMPARISON_TABLE_CSV}")

    n_complete = sum(1 for e in RESULTS["stages"].values() if e["status"] == "complete")
    n_failed = sum(1 for e in RESULTS["stages"].values() if e["status"] == "failed")
    print(f"\nStage summary: {n_complete} complete, {n_failed} failed, {len(RESULTS['stages'])} total.")
    if n_failed:
        print("\nFailed stages (see RESULTS['stages'][name]['traceback'] for detail):")
        for k, e in RESULTS["stages"].items():
            if e["status"] == "failed":
                print(f"  - {k}: {e.get('error')}")
    return df


comparison_df = build_comparison_table()
comparison_df


### Reading the results

- **Metric A rows**: per-session, per-color hit-rate @20mm for a zero-shot grounding prompt.
  Per plan §3, look at the **distribution across sessions**, not a single pooled number --
  and remember 4 of the 10 sessions are missing at least one color command (plan §3's table),
  so a `None`/`NaN` `hit_rate_<color>` for those sessions is expected, not a bug.
- **Metric B rows**: mean Euclidean error (mm) on held-out sessions, per candidate, per
  `variant` (`vision_only` vs `vision_plus_state`), per `n_future` (frames-ahead, from the
  `N_FUTURE_SWEEP`). Compare `vision_only` vs `vision_plus_state` at the same `n_future` to
  answer "does state help" without conflating it with "which backbone is better" (plan §2.2).
- Per plan §2.3: neither metric predicts final closed-loop control quality. Read this table as
  "which backbone is worth investing further engineering time in," never as "which arm wins."
- Per plan §3's flagged limitation: all 10 sessions were collected the same day/rig/lighting --
  read any ranking here as bootstrap signal under that one condition, not a robustness claim.
